In [1]:
import os

current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
os.chdir(parent_dir)
print("Current working directory:", os.getcwd())

Current working directory: /Users/alvarovilladangos/Desktop/TradingBot/Forex


In [ ]:
import time 

from ibkr.app import IBApp
from ibkr.contract import stock, future
from ibkr.order import market, BUY, SELL

app = IBApp("127.0.0.1", 7497, client_id=375, account="DUE310707")

psx = stock("PSX", "SMART", "USD")
ho = future("HO", "NYMEX", "202503")
rb = future("RB", "NYMEX", "202503")
cl = future("CL", "NYMEX", "202503")

window = 60
thresh = 2

while True:
    data = app.get_historical_data_for_many(
        request_id=99,
        contracts=[psx, ho, rb, cl],
        duration="1 D",
        bar_size="1 min",
    ).dropna()

    data["crack_spread"] = data.HO + 2 * data.RB - 3 * data.CL
    data["crack_spread_rank"] = data.crack_spread.rolling(window).rank(pct=True)
    data["refiner_rank"] = data.PSX.rolling(window).rank(pct=True)
    data["rank_spread"] = data.refiner_rank - data.crack_spread_rank

    roll = data.rank_spread.rolling(window)
    zscore = (data.rank_spread - roll.mean()) / roll.std()
    signal = zscore[-1]

    holding = psx.symbol in app.positions.keys()

    if signal <= -thresh and not holding:
        order = market(BUY, 10)
        app.send_order(psx, order)
    elif signal >= 0 and holding:
        app.order_target_percent(psx, market, 0)

    if signal >= thresh and not holding:
        order = market(SELL, 10)
        app.send_order(psx, order)
    elif signal <= 0 and holding:
        app.order_target_percent(psx, market, 0)

time.sleep(60)

app.disconnect()

ERROR -1 2104 La conexión al centro de datos funciona correctamente:usfarm.nj
ERROR -1 2104 La conexión al centro de datos funciona correctamente:cashfarm
ERROR -1 2104 La conexión al centro de datos funciona correctamente:usfarm
ERROR -1 2106 La conexión al centro de datos HMDS funciona correctamente:euhmds
ERROR -1 2106 La conexión al centro de datos HMDS funciona correctamente:cashhmds
ERROR -1 2106 La conexión al centro de datos HMDS funciona correctamente:fundfarm
ERROR -1 2106 La conexión al centro de datos HMDS funciona correctamente:ushmds
ERROR -1 2158 La conexión de la granja de datos "sec-def" funciona correctamente:secdefnj
/Users/alvarovilladangos/Desktop/TradingBot/Forex/ibkr/client.py:489: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  self.portfolio_returns = self.returns.pct

In [3]:
data

symbol,CL,HO,PSX,RB,crack_spread,crack_spread_rank,refiner_rank,rank_spread
time,,,,,,,,
2024-12-20 09:30:00-05:00,68.33,2.2022,110.62,1.9474,-198.8930,NaN,NaN,NaN
2024-12-20 09:31:00-05:00,68.26,2.2008,110.67,1.9455,-198.6882,NaN,NaN,NaN
2024-12-20 09:32:00-05:00,68.27,2.2013,110.72,1.9455,-198.7177,NaN,NaN,NaN
2024-12-20 09:33:00-05:00,68.23,2.2004,110.73,1.9447,-198.6002,NaN,NaN,NaN
2024-12-20 09:34:00-05:00,68.22,2.2005,110.94,1.9448,-198.5699,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2024-12-20 15:55:00-05:00,69.08,2.2266,110.18,1.9697,-201.0740,0.666667,0.183333,-0.483333
2024-12-20 15:56:00-05:00,69.04,2.2253,110.19,1.9687,-200.9573,0.750000,0.200000,-0.550000
2024-12-20 15:57:00-05:00,69.05,2.2256,110.18,1.9687,-200.9870,0.750000,0.191667,-0.558333


In [4]:
app.get_historical_data_for_many(
        request_id=99,
        contracts=[psx, ho, rb, cl],
        duration="1 D",
        bar_size="1 min",
    )

ValueError: Index contains duplicate entries, cannot reshape